# M5 · 검색 근거를 붙여 답하기

①·⑥·⑧ 문서에서 관련 내용을 검색하고, LangChain LCEL 체인으로 근거 판정·프롬프트·LLM 답변·인용 번호를 연결한다. 현재 Chroma 검색은 문자 n-gram 기준선이며 OpenAI 의미 임베딩으로 재색인한 결과가 아니다.

In [ ]:
from pathlib import Path
import hashlib,json
import numpy as np
import chromadb
from sklearn.feature_extraction.text import HashingVectorizer
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None,'SANUP-P 프로젝트 폴더에서 실행하세요.'
PERSONAL=ROOT/'data'/'personal'
CORPUS=PERSONAL/'corpus';DB=ROOT/'chroma_db'/'personal';DB.mkdir(parents=True,exist_ok=True)
vectorizer=HashingVectorizer(analyzer='char',ngram_range=(2,4),n_features=1024,
                             alternate_sign=False,norm='l2',dtype=np.float32)
def vectors(texts):return vectorizer.transform(texts).toarray().astype(np.float32)

from dotenv import dotenv_values
from openai import APIStatusError,OpenAIError
from langchain_openai import ChatOpenAI
from src.rag_chain import build_rag_chain

## 색인 연결과 문서 검색

In [ ]:
meta=json.loads((CORPUS/'index_meta.json').read_text(encoding='utf-8'))
client=chromadb.PersistentClient(path=str(DB))
collection=client.get_collection(meta['collection'],embedding_function=None)
assert collection.count()==meta['chunks']
def retrieve(question,k=5,source_type=None):
    where={'source_type':source_type} if source_type else None
    found=collection.query(query_embeddings=vectors([question]).tolist(),
                           n_results=min(k*8,collection.count()),where=where,
                           include=['documents','metadatas','distances'])
    hits=[];seen=set()
    for text,doc,distance in zip(found['documents'][0],found['metadatas'][0],found['distances'][0]):
        if doc['doc_id'] in seen:continue
        seen.add(doc['doc_id'])
        hits.append({'doc_id':doc['doc_id'],'text':text,'metadata':doc,'similarity':1-float(distance)})
        if len(hits)>=k:break
    return hits

## LangChain LCEL로 검색·근거 판정·생성·인용 확인 연결

In [ ]:
def ask_with_sources(question,source_type=None,min_similarity=0.12):
    config=dotenv_values(ROOT/'.env')
    key=(config.get('OPENAI_API_KEY') or '').strip()
    model=None
    if key:
        model=ChatOpenAI(model=(config.get('OPENAI_MODEL') or 'gpt-5-mini'),
                         api_key=key,timeout=60,max_retries=0,
                         max_completion_tokens=1200,reasoning_effort="low")
    chain=build_rag_chain(retrieve,model,min_similarity=min_similarity)
    try:
        return chain.invoke({'question':question,'source_type':source_type})
    except APIStatusError as exc:
        return {'answer':None,'sources':[], 'status':f'llm_http_{exc.status_code}'}
    except OpenAIError as exc:
        return {'answer':None,'sources':[], 'status':type(exc).__name__}

## 샘플 질문으로 검색 확인

In [ ]:
QUESTION='이동식 사다리 작업 전에 무엇을 점검해야 하나요?'
hits=retrieve(QUESTION,k=5,source_type='kosha_guide')
display([{'doc_id':h['doc_id'],'title':h['metadata']['title'],
          'score':round(h['similarity'],3),'url':h['metadata']['source_url']}
         for h in hits])
RUN_LLM_TEST=True  # 실행할 때 실제 API 호출 요금이 발생한다.
if RUN_LLM_TEST:
    result=ask_with_sources(QUESTION,source_type='kosha_guide')
    print('상태:',result['status'])
    print('답변:',result['answer'])
    print('인용 문서:',[(h['doc_id'],h['metadata']['source_url'],h['metadata']['page'])
                   for h in result['sources']])
else:
    print('검색 결과만 확인했다. LLM 답변은 실행할 때 검증한다.')

## RAG 구현 기록

- M5에서 Chroma의 6,731개 문자 기반 청크를 검색하고 LangChain LCEL로 근거 판정 → 프롬프트 → ChatOpenAI → 인용 번호 확인을 연결했다. 설치만 한 상태는 아니다.
- 2026-10-01에 새 키로 채팅과 OpenAI 임베딩을 각각 소량 호출해 성공했다. 임베딩 응답은 1,536차원이었다. 다만 **현재 Chroma 색인은 문자 n-gram 기준선**이므로, 임베딩 API 연결 성공을 의미 벡터 검색 완료로 해석하지 않는다.
- 실제 질문 “이동식 사다리 작업 전에 무엇을 점검해야 하나요?”에 앞선 실행에서 `answered`가 반환됐고, 인용 문서 `A-G-4-2025`의 PDF 7쪽에서 설치 바닥·전도 방지·최대사용하중·작업 높이·보호구 조건을 확인했다. 인용 번호와 원문 URL도 반환됐다.
- 무관한 “화성 탐사 로켓 연료” 질문에는 `insufficient_evidence`와 출처 0건이 반환됐다. 검색 유사도 0.12는 임시 기준이며, 일부 무관한 질문도 그 기준을 넘으므로 “모름” 응답의 일반 성능은 별도 질문 세트로 평가해야 한다.
- M6·M7 점수는 기존 문자 검색 색인의 결과다. 의미 임베딩으로 전체 문서를 재색인하고 같은 질문으로 비교하는 작업과 Streamlit 통합은 남아 있다.
- 재실행 때 모델 응답 본문이 비어 `empty_response`가 나온 사례가 있어 생성 토큰 상한을 450에서 1,200으로 높이고 추론 강도를 낮춰 재검증했다. 검색 성공과 답변 성공을 따로 확인한다.
- 재검증 과정에서 한 번은 검색 문서를 찾았지만 생성 본문이 비어 있었다. 출력 토큰 상한을 늘리고 추론 강도를 낮춘 뒤 `answered`가 다시 나왔다. 따라서 검색 성공과 생성 성공은 다른 검증 항목으로 본다.
- 그 답변에는 인용 없는 “사다리 손상 점검” 항목이 섞였다. 지금은 프롬프트에 근거 밖 항목 생략을 명시하고 목록 항목마다 끝에 인용이 없으면 `unsupported_citation`으로 거절한다. 인용 번호가 있다고 주장 자체가 원문에 맞는지까지 자동 보장하지는 않는다.

- 같은 질문의 다음 답변에서는 PDF 추출 텍스트에서 숫자 위치가 틀어져 `개 이상의 버팀대`라는 불완전한 수량이 인용과 함께 나왔다. 이 형태를 `malformed_quantity`로 거절하도록 했다. 다른 수치 표현 전체를 검증한 것은 아니다.


## RAG 체인 시험 결과

- 질문에 맞춰 근거를 검색하고, 검색된 문서만 답변 근거로 쓰도록 체인을 연결했다. 인용 번호는 실제 검색 결과의 문서 ID·제목·URL·페이지와 연결되며, 근거가 부족하면 모른다고 답한다.
- 이동식 사다리 질문의 시험에서는 KOSHA GUIDE `A-G-4-2025` 7쪽을 찾아 설치 바닥, 전도 방지, 최대사용하중, 작업 높이 등 지침에 적힌 내용을 인용했다. 해당 답은 자료에 적힌 내용이지 현장 위험성평가를 대체하지 않는다.
- 자동 단위 테스트는 근거 없는 문장, 인용 누락, 잘못된 수량 표현, 근거 부족 시 답변 거절, TBM 시 두 자료 유형을 함께 인용하는 조건을 검사한다. 현재 15개 테스트가 통과했다.
- 테스트 질문 1건의 성공만으로 답변 전체가 사실에 맞는다고 결론 낼 수 없다. 실제 사용자는 인용된 공식 페이지와 PDF 페이지를 열어 문구를 확인해야 한다.

### 운영상 유의사항

- LLM 응답은 검색 청크가 틀리거나 OCR이 잘못 읽으면 오류를 만들 수 있다. 출처 URL과 페이지를 사용자에게 보여주고 OCR 여부도 같이 표시한다.
- API 키·질문·작업 맥락 전송 여부를 확인하고, 민감한 현장정보는 입력하지 않는다.
- 평가는 M6 정답 문서 검색 점수와 M8 제한된 답변 표본 점검으로 나뉜다. 검색 점수가 답변의 안전성·정확성을 그대로 뜻하지 않는다.
